In [ ]:
from pathlib import Path
ROOT=Path.cwd()
if ROOT.name=='src':ROOT=ROOT.parent
if not (ROOT/'data').is_dir():raise FileNotFoundError('Open this notebook from Code or Code/src.')
import gc
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize,TwoSlopeNorm
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import FormatStrFormatter

PROCESSED_DIR=ROOT/'data/processed'
BOUNDARY_DIR=ROOT/'external/boundaries'
OUTPUT_DIR=ROOT/'output'
FIG=OUTPUT_DIR/'figures'
TAB=OUTPUT_DIR/'tables'
BOUNDARY=BOUNDARY_DIR
plt.rcParams.update({'pdf.fonttype':42,'ps.fonttype':42,'font.family':'DejaVu Sans','font.size':10})

MONTHS={1:'Jan.',2:'Feb.',3:'Mar.',4:'Apr.',5:'May',6:'Jun.',7:'Jul.',8:'Aug.',9:'Sep.',10:'Oct.',11:'Nov.',12:'Dec.'}
SEASONS={12:'Winter',1:'Winter',2:'Winter',3:'Spring',4:'Spring',5:'Spring',6:'Summer',7:'Summer',8:'Summer',9:'Fall',10:'Fall',11:'Fall'}
REGIONS={}
for region,states in {'Northeast':[9,23,25,33,44,50,34,36,42],
    'Midwest':[17,18,26,39,55,19,20,27,29,31,38,46],
    'South':[10,11,12,13,24,37,45,51,54,1,21,28,47,5,22,40,48],
    'West':[4,8,16,30,32,35,49,56,2,6,15,41,53]}.items():
    REGIONS.update({state:region for state in states})

In [ ]:
def load_plot_data():
    monthly = pd.read_parquet(PROCESSED_DIR/'regional_monthly_summary.parquet')
    annual = pd.read_parquet(PROCESSED_DIR/'annual_summary.parquet')
    maps = pd.read_parquet(PROCESSED_DIR/'figure2_tract_month.parquet')
    return monthly, annual, maps


In [ ]:
def table(name,df):
    from openpyxl.styles import Font,Border,Side,Alignment
    from openpyxl.utils import get_column_letter
    TAB.mkdir(parents=True,exist_ok=True)
    path=TAB/f'Table {name}.xlsx'
    with pd.ExcelWriter(path,engine='openpyxl') as writer:
        df.to_excel(writer,sheet_name='Sheet1',index=False)
        sheet=writer.sheets['Sheet1']
        sheet.sheet_view.showGridLines=False
        rule=Side(style='thin',color='FF000000')
        for row in sheet:
            for cell in row:
                cell.font=Font(name='Times New Roman',size=10,bold=cell.row==1)
                cell.alignment=Alignment(vertical='center',horizontal='center' if cell.row==1 else ('right' if isinstance(cell.value,(int,float)) else 'left'),wrap_text=cell.row==1)
                if isinstance(cell.value,float):cell.number_format='0.00'
                if cell.row==1:cell.border=Border(top=rule,bottom=rule)
                elif cell.row==sheet.max_row:cell.border=Border(bottom=rule)
        sheet.row_dimensions[1].height=36
        for i in range(2,sheet.max_row+1):sheet.row_dimensions[i].height=18
        for j,column in enumerate(df.columns,1):
            sheet.column_dimensions[get_column_letter(j)].width=14 if column in ['Month','Year','Region','Season'] else 27

def save_fig(fig,name):
    FIG.mkdir(parents=True,exist_ok=True)
    for ext in ['png','pdf']:
        p=FIG/f'Figure2.{ext}'
        fig.savefig(p,dpi=600,bbox_inches='tight')
    plt.close(fig)

def stats_tables(monthly, yearly):
    years = [2009, 2019, 2020, 2022]
    names = {y: f'Monthly mean, {y} (kWh)' for y in years}
    sel = monthly[monthly.YEAR.isin(years)].copy()
    national = sel.groupby(['MONTH', 'YEAR'])[['total_predicted_kwh', 'tract_count']].sum()
    s2 = (national.total_predicted_kwh / national.tract_count).unstack('YEAR').reindex(columns=years)
    s2.insert(0, 'Month', s2.index.map(MONTHS))
    s2 = s2.rename(columns=names).reset_index(drop=True)

    def diff(base, keys):
        out = base[keys].copy()
        for a, b in [(2019, 2009), (2022, 2020)]:
            out[f'Absolute change, {a} minus {b} (kWh)'] = base[names[a]] - base[names[b]]
            out[f'Relative change, {a} vs. {b} (%)'] = 100 * (base[names[a]] - base[names[b]]) / base[names[b]]
        return out
    sel['Season'] = sel.MONTH.map(SEASONS)
    regional = sel.groupby(['Region', 'Season', 'YEAR'])[['total_predicted_kwh', 'tract_count']].sum()
    s4 = (regional.total_predicted_kwh / regional.tract_count).unstack('YEAR').reindex(columns=years).rename(columns=names).reset_index()
    s4.Region = pd.Categorical(s4.Region, ['Northeast', 'Midwest', 'South', 'West'], ordered=True)
    s4.Season = pd.Categorical(s4.Season, ['Winter', 'Spring', 'Summer', 'Fall'], ordered=True)
    s4 = s4.sort_values(['Region', 'Season']).reset_index(drop=True)
    s6 = yearly[['YEAR']].rename(columns={'YEAR': 'Year'}).copy()
    s6['Mean annual total (kWh)'] = yearly.total_annual_predicted_kwh / yearly.tract_count
    s6['Absolute YoY change (kWh)'] = s6['Mean annual total (kWh)'].diff()
    s6['Relative YoY change (%)'] = 100 * s6['Mean annual total (kWh)'].pct_change()
    for name, data in {'S2': s2, 'S3': diff(s2, ['Month']), 'S4': s4, 'S5': diff(s4, ['Region', 'Season']), 'S6': s6}.items():
        table(name, data)


In [ ]:
def basemap(year, projection, clip=True):
    tr = gpd.read_file(BOUNDARY/f'tract_{year}.shp')[['GEOID','geometry']]
    st = gpd.read_file(BOUNDARY/f'state_{year}.shp')[['STATEFP','geometry']]
    tr.GEOID=pd.to_numeric(tr.GEOID).astype('int64')
    tr=tr.to_crs(4326); st=st.to_crs(4326)
    if clip:
        extent=gpd.GeoDataFrame(geometry=[box(-125,24,-66.5,49.5)],crs=4326)
        tr=gpd.overlay(tr,extent,how='intersection')
        st=gpd.overlay(st,extent,how='intersection').dissolve(by='STATEFP')
    else:
        tr=tr[~(tr.GEOID//1000000000).isin([2,15,60,66,69,72,78])]
        st=st[~pd.to_numeric(st.STATEFP).isin([2,15,60,66,69,72,78])]
    return tr.to_crs(projection),st.to_crs(projection)

def figure2(df):
    maps={}
    for y in [2019,2020]:
        maps[y]=basemap(y,5070)
    d=df[df.MONTH.isin([1,4,7,10]) & df.YEAR.isin([2009,2019,2020,2022])]
    pieces=[]
    for month in [1,4,7,10]:
        dm=d[d.MONTH.eq(month)].pivot(index='GEOID',columns='YEAR',values='PRED_ELEC_M')
        for col,year,value in [(0,2019,dm[2009]),(1,2019,dm[2019]-dm[2009]),(2,2020,dm[2020]),(3,2020,dm[2022]-dm[2020])]:
            value=value.dropna().rename('value')
            joined=maps[year][0].merge(value,on='GEOID',how='left',validate='many_to_one')
            pieces.append((month,col,year,joined))
    absvals=np.concatenate([g.value.dropna().to_numpy() for m,c,y,g in pieces if c in [0,2]])
    diffvals=np.concatenate([g.value.dropna().to_numpy() for m,c,y,g in pieces if c in [1,3]])
    vmin,vmax=float(absvals.min()),float(absvals.max()); delta=float(abs(diffvals).max())
    normdiff=TwoSlopeNorm(vmin=-delta,vcenter=0,vmax=delta)
    fig,axs=plt.subplots(4,4,figsize=(16,13))
    for idx,(month,col,year,g) in enumerate(pieces):
        row=idx//4; ax=axs[row,col]; ax.set_axis_off()
        opts={'cmap':'viridis','vmin':vmin,'vmax':vmax} if col in [0,2] else {'cmap':'RdBu_r','norm':normdiff}
        g.plot(column='value',ax=ax,linewidth=0,edgecolor='none',rasterized=True,**opts)
        maps[year][1].boundary.plot(ax=ax,color='black',linewidth=.50,alpha=1,zorder=6)
        titles=[f'{MONTHS[month]} 2009',f'Difference in {MONTHS[month]} 2019–2009',f'{MONTHS[month]} 2020',f'Difference in {MONTHS[month]} 2022–2020']
        ax.set_title(titles[col],fontsize=11,loc='center',pad=8)
        ax.text(.01,1.04,'abcdefghijklmnop'[idx],transform=ax.transAxes,ha='left',va='bottom',fontsize=14,fontweight='bold',zorder=7)
        ax.margins(0)
    ca=fig.add_axes([.10,.035,.35,.018]); cd=fig.add_axes([.55,.035,.35,.018])
    cb=fig.colorbar(ScalarMappable(norm=Normalize(vmin,vmax),cmap='viridis'),cax=ca,orientation='horizontal',format=FormatStrFormatter('%.2f'))
    cb.set_label('Monthly Electricity Consumption (kWh)')
    cb2=fig.colorbar(ScalarMappable(norm=normdiff,cmap='RdBu_r'),cax=cd,orientation='horizontal',format=FormatStrFormatter('%.2f'))
    cb2.set_label('Difference in Monthly Electricity Consumption (kWh)')
    for c in [cb,cb2]:
        for s in c.ax.spines.values(): s.set_visible(False)
    fig.subplots_adjust(left=.04,right=.98,top=.96,bottom=.09,wspace=.03,hspace=.01)
    save_fig(fig,'Figure2')
    del maps,pieces
    gc.collect()


In [ ]:
monthly_summary, annual_summary, map_values = load_plot_data()
stats_tables(monthly_summary, annual_summary)
figure2(map_values)
